# LumenY — 04: Probability Calibration

Fits an isotonic regression calibrator on the derived P(down) probabilities.

**What we calibrate:** The `np.interp()` derived probability — not the quantile models themselves.

**Splits:**
- Train: 2009-2020 → quantile models already trained on this
- Calibration: 2020-2022 → fit isotonic regressor here
- Test: 2022-2025 → final honest bucket table reported here only

**Output:** One isotonic calibrator per horizon saved to `backend/models/`

In [ ]:
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
from sklearn.isotonic import IsotonicRegression

FEATURES_DIR = Path('../backend/data/features')
MODELS_DIR   = Path('../backend/models')

HORIZONS       = ['1H', '4H', '1D', '7D']
QUANTILES      = [0.10, 0.25, 0.50, 0.75, 0.90]
QUANTILE_NAMES = ['Q10', 'Q25', 'Q50', 'Q75', 'Q90']

print('Ready.')

## 1. Load Dataset and Define Splits

In [ ]:
df = pd.read_parquet(FEATURES_DIR / 'all_pairs_features_labels.parquet')

label_cols   = [f'label_{h}' for h in HORIZONS]
drop_cols    = label_cols + ['pair']
feature_cols = [c for c in df.columns if c not in drop_cols]

# Define splits by date
calib_start = '2020-01-01'
calib_end   = '2022-01-01'
test_start  = '2022-01-01'

df_calib = df[(df.index >= calib_start) & (df.index < calib_end)]
df_test  = df[df.index >= test_start]

print(f'Full dataset:  {len(df):,} rows')
print(f'Calibration:   {len(df_calib):,} rows ({calib_start} to {calib_end})')
print(f'Test:          {len(df_test):,} rows ({test_start} to end)')

## 2. Probability Derivation Function

Improved version — extrapolates beyond Q10/Q90 instead of hard-capping at 0.01/0.99.

In [ ]:
def derive_p_down(q_vals: np.ndarray) -> float:
    """
    Derive P(down) from 5 quantile predictions [Q10, Q25, Q50, Q75, Q90].
    
    Improved: extrapolates beyond Q10/Q90 using the tail slope
    instead of hard-capping at 0.01/0.99.
    """
    qs   = np.array(QUANTILES)    # [0.10, 0.25, 0.50, 0.75, 0.90]
    vals = q_vals                  # [Q10_val, Q25_val, Q50_val, Q75_val, Q90_val]
    
    # If 0 is within the range of predicted quantiles — interpolate directly
    if vals[0] <= 0 <= vals[-1]:
        return float(np.interp(0, vals, qs))
    
    # If all quantiles are negative — extrapolate beyond Q90 using Q75-Q90 slope
    elif vals[-1] < 0:
        slope = (qs[-1] - qs[-2]) / (vals[-1] - vals[-2] + 1e-10)
        p_down = qs[-1] + slope * (0 - vals[-1])
        return float(np.clip(p_down, 0.90, 0.999))
    
    # If all quantiles are positive — extrapolate beyond Q10 using Q10-Q25 slope
    else:
        slope = (qs[1] - qs[0]) / (vals[1] - vals[0] + 1e-10)
        p_down = qs[0] + slope * (0 - vals[0])
        return float(np.clip(p_down, 0.001, 0.10))


def get_all_p_downs(df_subset, feature_cols, horizon):
    """
    Run all 5 quantile models on a dataset and derive P(down) for each row.
    Returns (p_downs, y_actual)
    """
    X = df_subset[feature_cols].ffill().bfill()
    y = df_subset[f'label_{horizon}'].values
    
    # Load and run all 5 quantile models
    q_preds = {}
    for q, q_name in zip(QUANTILES, QUANTILE_NAMES):
        bundle = joblib.load(MODELS_DIR / f'model_{horizon}_Q{int(q*100)}.joblib')
        q_preds[q_name] = bundle['model'].predict(X)
    
    # Derive P(down) for each row
    p_downs = np.array([
        derive_p_down(np.array([q_preds[n][i] for n in QUANTILE_NAMES]))
        for i in range(len(X))
    ])
    
    return p_downs, y


print('Probability derivation functions ready.')

## 3. Fit Isotonic Calibrators on Calibration Set

In [ ]:
calibrators = {}

for horizon in HORIZONS:
    print(f'Fitting calibrator for {horizon}...')
    
    # Get raw probabilities on calibration set
    p_downs_calib, y_calib = get_all_p_downs(df_calib, feature_cols, horizon)
    
    # Binary label: 1 if price went down
    y_binary = (y_calib < 0).astype(float)
    
    # Fit isotonic regression
    # increasing=True means: higher raw probability → higher calibrated probability
    iso = IsotonicRegression(out_of_bounds='clip', increasing=True)
    iso.fit(p_downs_calib, y_binary)
    
    calibrators[horizon] = iso
    
    # Quick check on calibration set itself
    p_cal = iso.predict(p_downs_calib)
    print(f'  Calibration set size: {len(p_downs_calib):,}')
    print(f'  Raw P(down) range:        [{p_downs_calib.min():.3f}, {p_downs_calib.max():.3f}]')
    print(f'  Calibrated P(down) range: [{p_cal.min():.3f}, {p_cal.max():.3f}]')
    
    # Save calibrator
    joblib.dump(iso, MODELS_DIR / f'calibrator_{horizon}.joblib')
    print(f'  Saved: calibrator_{horizon}.joblib')

print('\nAll calibrators fitted and saved.')

## 4. Evaluate on Test Set (2022-2025)

This is the honest final evaluation — data the calibrator has never seen.

In [ ]:
def bucket_analysis(p_downs_raw, p_downs_cal, y_actual, horizon):
    """
    Compare raw vs calibrated bucket accuracy on test set.
    """
    correct_raw = np.where(p_downs_raw > 0.5, y_actual < 0, y_actual > 0)
    correct_cal = np.where(p_downs_cal > 0.5, y_actual < 0, y_actual > 0)
    
    p_dom_raw = np.where(p_downs_raw > 0.5, p_downs_raw, 1 - p_downs_raw)
    p_dom_cal = np.where(p_downs_cal > 0.5, p_downs_cal, 1 - p_downs_cal)
    
    buckets = [(0.50, 0.55), (0.55, 0.60), (0.60, 0.65),
               (0.65, 0.70), (0.70, 0.80), (0.80, 0.90), (0.90, 1.00)]
    
    print(f'\nHorizon: {horizon}')
    print(f'  {"Bucket":<12} {"Raw pred":<12} {"Raw actual":<13} {"Cal pred":<12} {"Cal actual":<13} {"Count":<8} {"Status"}')
    print(f'  {"-"*80}')
    
    for low, high in buckets:
        mask_raw = (p_dom_raw >= low) & (p_dom_raw < high)
        mask_cal = (p_dom_cal >= low) & (p_dom_cal < high)
        
        if mask_cal.sum() < 20:
            continue
        
        raw_pred   = p_dom_raw[mask_raw].mean() if mask_raw.sum() > 0 else 0
        raw_actual = correct_raw[mask_raw].mean() if mask_raw.sum() > 0 else 0
        cal_pred   = p_dom_cal[mask_cal].mean()
        cal_actual = correct_cal[mask_cal].mean()
        count      = mask_cal.sum()
        gap        = abs(cal_actual - cal_pred)
        status     = 'OK' if gap < 0.05 else 'NEEDS WORK'
        
        print(f'  {low:.0%}-{high:.0%}      '
              f'{raw_pred:.1%}        {raw_actual:.1%}         '
              f'{cal_pred:.1%}        {cal_actual:.1%}         '
              f'{count:<8} {status}')


print('Running bucket analysis on TEST SET (2022-2025)...\n')
print('=' * 80)

test_results = {}

for horizon in HORIZONS:
    # Get raw probabilities on test set
    p_downs_raw, y_test = get_all_p_downs(df_test, feature_cols, horizon)
    
    # Apply calibrator
    iso = calibrators[horizon]
    p_downs_cal = iso.predict(p_downs_raw)
    
    bucket_analysis(p_downs_raw, p_downs_cal, y_test, horizon)
    test_results[horizon] = (p_downs_raw, p_downs_cal, y_test)

## 5. Calibration Curves — Before vs After

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
fig.patch.set_facecolor('#080c14')

for col, horizon in enumerate(HORIZONS):
    p_raw, p_cal, y_test = test_results[horizon]
    y_binary = (y_test < 0).astype(float)
    
    for row, (p_vals, title) in enumerate([(p_raw, 'Raw'), (p_cal, 'Calibrated')]):
        ax = axes[row][col]
        ax.set_facecolor('#080c14')
        
        # Bucket the probabilities
        bins = np.linspace(0, 1, 11)
        bin_preds, bin_actuals = [], []
        
        for i in range(len(bins)-1):
            mask = (p_vals >= bins[i]) & (p_vals < bins[i+1])
            if mask.sum() > 30:
                bin_preds.append(p_vals[mask].mean())
                bin_actuals.append(y_binary[mask].mean())
        
        ax.plot([0,1],[0,1],'--',color=(1,1,1,0.2),linewidth=1)
        ax.plot(bin_preds, bin_actuals, 'o-', color='#4fc3f7', linewidth=2, markersize=6)
        ax.set_xlim(0,1); ax.set_ylim(0,1)
        ax.set_title(f'{horizon} — {title}', color='white', fontsize=10)
        ax.tick_params(colors='white', labelsize=7)
        for spine in ax.spines.values(): spine.set_edgecolor('#1a2332')
        
        if bin_preds:
            mce = np.mean(np.abs(np.array(bin_preds) - np.array(bin_actuals)))
            ax.text(0.05, 0.92, f'MCE: {mce:.3f}', transform=ax.transAxes,
                    color='#4fc3f7', fontsize=8)

plt.suptitle('Calibration: Raw vs Isotonic — Test Set 2022-2025', color='white', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# Final summary
print('=' * 55)
print('CALIBRATION COMPLETE')
print('=' * 55)
print(f'\nCalibrators saved:')
for f in sorted(MODELS_DIR.glob('calibrator_*.joblib')):
    print(f'  {f.name}')
print(f'\nInference pipeline:')
print(f'  1. Run 5 quantile models → Q10, Q25, Q50, Q75, Q90')
print(f'  2. Derive raw P(down) via extrapolated interpolation')
print(f'  3. Apply isotonic calibrator → calibrated P(down)')
print(f'  4. Output to UI')